<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/ML_4DVAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')

Mounted at /content/drive


# ML Emulator

In [ ]:
# ============================================================
# train_rescnn_1layer_configurable_ai4dvar2_keweak.py
# ------------------------------------------------------------
# Configurable residual-CNN trainer for 1-layer SWE emulator
# on Klein beta-plane, with:
#   - multistep rollout data loss
#   - collocation state loss
#   - collocation tendency loss
#   - continuity residual (flux form)
#   - momentum residual (nonlinear)
#   - weak geostrophic-balance loss
#   - weak damping / smoothness loss
#   - NEW: weak domain-mean KE constraint
#
# Outputs go to:
#   /content/drive/MyDrive/AI_4DVAR2/...
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------
class CFG:
    # -------- paths --------
    ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
    ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

    DATA_DIR_CANDIDATES = [
        f"{ROOT_IN}/klein_ckpt_1L",
        f"{ROOT_IN}/klein_ckpt_AL_centers_colloc",
        f"{ROOT_IN}/klein_ckpt_AL_centers",
        f"{ROOT_IN}/klein_ckpt_1L_colloc",
    ]
    COLLOC_DIR = f"{ROOT_IN}/klein_ml_1L/colloc_raw"

    # customize this tag per experiment
    # EXP_NAME = "rescnn_b8_c96_lr1e4_t6_p4_keweak"
    EXP_NAME = "rescnn_b12_c96_lr1e4_t6_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t8_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t8_p4_keweak_lam001"
    # EXP_NAME = "rescnn_b12_c128_lr1e4_t8_p4_keweak"
    # EXP_NAME = "rescnn_b12_c96_lr1e4_t6_p4_keweak_lam001"
    # EXP_NAME = "rescnn_b12_c128_lr1e4_t6_p4_keweak"

    # -------- domain / physics --------
    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 200.0 * 30.0
    FMIN = 2.0e-5

    # -------- architecture --------
    # N_BLOCKS = 8
    N_BLOCKS = 12
    FEAT_CH  = 96
    HIDDEN   = 192
    # FEAT_CH  = 128
    # HIDDEN   = 256

    # -------- optimization --------
    EPOCHS = 12
    BATCH_SIZE = 1
    LR = 1e-4
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0

    # -------- rollout --------
    ROLL_STEPS = 4
    ROLL_GAMMA = 0.95

    # -------- data loading --------
    MAX_WINDOWS_PER_IC = 4
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # -------- collocation controls --------
    # N_TIME_SLICES = 8
    N_TIME_SLICES = 6
    PTS_PER_TIME  = 4

    # -------- loss weights --------
    LAMBDA_DATA       = 1.0
    LAMBDA_COLL_STATE = 0.05
    LAMBDA_COLL_TEND  = 0.1
    LAMBDA_CONT       = 0.2
    LAMBDA_MOM        = 0.5
    LAMBDA_GEO        = 0.01
    LAMBDA_SMOOTH     = 1e-4

    # -------- NEW: weak KE loss --------
    # Uses log-domain mismatch of domain-mean KE over rollout.
    # Start weak. If too weak, try 0.05; if unstable, try 0.005.
    LAMBDA_KE = 0.02
    KE_EPS = 1e-6

    # -------- misc --------
    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 10
    RESUME_FROM = None

cfg = CFG()

cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
            "exp_name": cfg.EXP_NAME,
            "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom",
            "train_geo", "train_smooth",
            "train_ke"
        ])
        for row in loss_history:
            w.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )



        # zero-init safe start
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True,
        )
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        return state0_local + tau.unsqueeze(-1) * delta

# ------------------------------------------------------------
# Collocation sampling
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Derivative / balance helpers
# ------------------------------------------------------------
def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)

# ------------------------------------------------------------
# NEW: domain-mean KE helper
# ------------------------------------------------------------
def domain_mean_ke_torch(state, h_floor=1.0):
    """
    state: [B, 3, NY, NX] with channels [eta, u, v]
    returns: [B] domain-mean KE
    """
    eta = state[:, 0]
    u   = state[:, 1]
    v   = state[:, 2]

    h = cfg.H + eta
    h_safe = torch.clamp(h, min=h_floor)

    ke = 0.5 * h_safe * (u * u + v * v)
    return ke.mean(dim=(-2, -1))

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / cfg.Lx) - 1.0
    y_norm = (2.0 * y_m / cfg.Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = cfg.H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / cfg.Lx)
    eta_y = eta_yn * (2.0 / cfg.Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / cfg.Lx)
    u_y = u_yn * (2.0 / cfg.Ly)
    v_x = v_xn * (2.0 / cfg.Lx)
    v_y = v_yn * (2.0 / cfg.Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x = hu_xn * (2.0 / cfg.Lx)
    hv_y = hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    # state collocation
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state += ((u_hat - u_true) ** 2).mean()
    loss_coll_state += ((v_hat - v_true) ** 2).mean()

    # tendency collocation
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend += ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend += ((v_t - dvc_dt_fd) ** 2).mean()

    # continuity in flux form
    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    # nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    # weak geostrophic balance penalty
    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g =  (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    # weak smoothness / damping penalty
    loss_smooth = (u_x ** 2 + u_y ** 2 + v_x ** 2 + v_y ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom, loss_geo, loss_smooth

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)

    if not os.path.exists(cfg.COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {cfg.COLLOC_DIR}")

    colloc_bank = CollocBank(cfg.COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=cfg.ROLL_STEPS,
        max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel(
        feat_ch=cfg.FEAT_CH,
        hidden=cfg.HIDDEN,
        n_blocks=cfg.N_BLOCKS,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    if cfg.RESUME_FROM is not None and os.path.exists(cfg.RESUME_FROM):
        print(f"[Resume] loading checkpoint: {cfg.RESUME_FROM}")
        loaded_epoch, loaded_history, _ = load_checkpoint(cfg.RESUME_FROM, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_BLOCKS              = {cfg.N_BLOCKS}")
    print(f"[Train] FEAT_CH               = {cfg.FEAT_CH}")
    print(f"[Train] LR                    = {cfg.LR}")
    print(f"[Train] N_TIME_SLICES         = {cfg.N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {cfg.PTS_PER_TIME}")
    print(f"[Train] colloc/interval       = {cfg.N_TIME_SLICES * cfg.PTS_PER_TIME}")
    print(f"[Train] LAMBDA_KE             = {cfg.LAMBDA_KE}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0
        run_geo = 0.0
        run_smooth = 0.0
        run_ke = 0.0

        for ib, batch in enumerate(loader):
            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = torch.tensor(0.0, device=device)
            loss_cstate = torch.tensor(0.0, device=device)
            loss_ctend = torch.tensor(0.0, device=device)
            loss_cont = torch.tensor(0.0, device=device)
            loss_mom = torch.tensor(0.0, device=device)
            loss_geo = torch.tensor(0.0, device=device)
            loss_smooth = torch.tensor(0.0, device=device)

            # NEW: KE loss
            loss_ke = torch.tensor(0.0, device=device)

            n_phys = 0

            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + cfg.DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                wk = (cfg.ROLL_GAMMA ** k)

                # main state loss
                loss_data = loss_data + wk * ((x_next - truth_next) ** 2).mean()

                # NEW: weak KE constraint in log space
                ke_pred = domain_mean_ke_torch(x_next)
                ke_true = domain_mean_ke_torch(truth_next)

                log_ke_pred = torch.log(ke_pred + cfg.KE_EPS)
                log_ke_true = torch.log(ke_true + cfg.KE_EPS)

                loss_ke = loss_ke + wk * ((log_ke_pred - log_ke_true) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=cfg.N_TIME_SLICES,
                        pts_per_time=cfg.PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_smooth = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=cfg.DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_smooth += l_smooth
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
                loss_geo = loss_geo / n_phys
                loss_smooth = loss_smooth / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero
                loss_geo = zero
                loss_smooth = zero

            loss = (
                cfg.LAMBDA_DATA         * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND  * loss_ctend
                + cfg.LAMBDA_CONT       * loss_cont
                + cfg.LAMBDA_MOM        * loss_mom
                + cfg.LAMBDA_GEO        * loss_geo
                + cfg.LAMBDA_SMOOTH     * loss_smooth
                + cfg.LAMBDA_KE         * loss_ke
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  data   =", float(loss_data.detach().cpu()))
                print("  cstate =", float(loss_cstate.detach().cpu()))
                print("  ctend  =", float(loss_ctend.detach().cpu()))
                print("  cont   =", float(loss_cont.detach().cpu()))
                print("  mom    =", float(loss_mom.detach().cpu()))
                print("  geo    =", float(loss_geo.detach().cpu()))
                print("  smooth =", float(loss_smooth.detach().cpu()))
                print("  ke     =", float(loss_ke.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.detach().cpu())
            run_data += float(loss_data.detach().cpu())
            run_cstate += float(loss_cstate.detach().cpu())
            run_ctend += float(loss_ctend.detach().cpu())
            run_cont += float(loss_cont.detach().cpu())
            run_mom += float(loss_mom.detach().cpu())
            run_geo += float(loss_geo.detach().cpu())
            run_smooth += float(loss_smooth.detach().cpu())
            run_ke += float(loss_ke.detach().cpu())

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} "
                    f"data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} "
                    f"ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} "
                    f"mom={float(loss_mom.detach().cpu()):.6e} "
                    f"geo={float(loss_geo.detach().cpu()):.6e} "
                    f"smooth={float(loss_smooth.detach().cpu()):.6e} "
                    f"ke={float(loss_ke.detach().cpu()):.6e}"
                )

        n_batches = max(len(loader), 1)
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches
        ep_geo = run_geo / n_batches
        ep_smooth = run_smooth / n_batches
        ep_ke = run_ke / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend,
            ep_cont, ep_mom, ep_geo, ep_smooth, ep_ke
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"geo={ep_geo:.6e} | smooth={ep_smooth:.6e} | "
            f"ke={ep_ke:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()

Using device: cuda
GPU: NVIDIA H100 80GB HBM3
[AutoDetect] checking candidate snapshot roots...
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
     valid IC dirs = 28, total pairs = 168
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers_colloc
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_AL_centers
     valid IC dirs = 0, total pairs = 0
  /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L_colloc
     valid IC dirs = 0, total pairs = 0
[AutoDetect] using: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
[CollocBank] loaded gauss_00 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_01 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_02 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_03 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_04 | samples=15616 | macro_groups=7
[CollocBank] loaded gauss_05 | samples=15616 | macro_groups=7
[CollocBank] loaded mix_00 | samples=15616 | macro_groups=7
[CollocBank] loaded m

KeyboardInterrupt: 

In [ ]:
# ML4DVAR V0

In [2]:
# ============================================================
# ml4dvar_lite_rescnn_1layer.py
# ------------------------------------------------------------
# Prototype ML-4DVAR using final 1-layer SWE emulator:
#   rescnn_b12_c96_lr1e4_t6_p4_keweak
#
# Goal:
#   optimize initial state x0 so that ML rollout matches
#   sparse synthetic observations over an assimilation window.
#
# State:
#   x = [eta, u, v]
#
# Output:
#   /content/drive/MyDrive/AI_4DVAR2/ml4dvar_lite_outputs
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR = f"{ROOT_IN}/klein_ckpt_1L"

CKPT_PATH = (
    f"{ROOT_OUT}/training_runs/"
    f"rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt"
)

OUT_DIR = f"{ROOT_OUT}/ml4dvar_lite_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
DT_MACRO = 200.0 * 30.0

NX = 256
NY = 128

# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
# IC_KEY = "wave_00"        # try also: "test_rh_00", "test_modon_00", "gauss_00"
# IC_KEY = "test_rh_00"        # try also: "test_rh_00", "test_modon_00", "gauss_00"
IC_KEY = "gauss_00"


START_INDEX = 0
WINDOW_STEPS = 4          # assimilation window length

OBS_FRAC = 0.03           # fraction of grid observed
OBS_NOISE_ETA = 0.0       # start noiseless
OBS_NOISE_UV  = 0.0

# N_OPT = 250                 # for 'wave_00'
N_OPT = 250                 # for 'test_rh_00'

# LR = 1e-2                 # for 'test_rh_00'
LR = 2e-2                   # for 'wave_00'

# LAMBDA_BG  = 1e-3         # background regularization
# LAMBDA_BG  = 5e-2         # background regularization
# LAMBDA_BG  = 0.10         # background regularization (for 'wave_00')
LAMBDA_BG  = 0.10         # background regularization (for 'test_rh_00')

#LAMBDA_SMOOTH = 1e-3      # smoothness on increment (for 'wave_00')
LAMBDA_SMOOTH = 1e-3      # smoothness on increment (for 'test_rh_00')

PERT_ETA_STD = 2.0        # background perturbation
PERT_UV_STD  = 0.2

PRINT_EVERY = 25

LAMBDA_UV_INC = 0.50

# ------------------------------------------------------------
# Model definition
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=12):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]

        blocks = []
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        # Needed because the trained checkpoint contains query_mlp weights.
        # The 4DVAR rollout does not use it, but the checkpoint expects it.
        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def forward_step(self, x):
        feat = self.encode(x)
        xdot = self.grid_tendency(feat)
        return x + DT_MACRO * xdot

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
_step_re = re.compile(r"klein_step_(\d+)\.npz")

def load_sequence(ic_key, start_index=0, window_steps=4):
    ic_dir = os.path.join(DATA_DIR, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    if len(files) < start_index + window_steps + 1:
        raise RuntimeError(f"Not enough snapshots for {ic_key}")

    seq = []
    times = []

    for f in files[start_index:start_index + window_steps + 1]:
        z = np.load(f)
        state = np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32)
        seq.append(state)
        times.append(float(z["t"]))

    seq = np.stack(seq, axis=0)
    times = np.array(times)

    return seq, times

def make_obs_mask(ny, nx, obs_frac, seed=123):
    rng = np.random.default_rng(seed)
    mask = rng.random((ny, nx)) < obs_frac
    return mask.astype(np.float32)

def smoothness_loss(dx):
    # dx: [1, 3, NY, NX]
    ddx = dx[:, :, :, 1:] - dx[:, :, :, :-1]
    ddy = dx[:, :, 1:, :] - dx[:, :, :-1, :]
    return (ddx ** 2).mean() + (ddy ** 2).mean()

def rmse_np(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def rollout_model(model, x0, steps):
    xs = [x0]
    x = x0
    for _ in range(steps):
        x = model.forward_step(x)
        xs.append(x)
    return xs

# ------------------------------------------------------------
# Load emulator
# ------------------------------------------------------------
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(CKPT_PATH)

model = MultiStepContinuousResCNNModel(
    feat_ch=96,
    hidden=192,
    n_blocks=12,
).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

for p in model.parameters():
    p.requires_grad_(False)

print("[Load] emulator:", CKPT_PATH)

# ------------------------------------------------------------
# Load truth and create synthetic obs
# ------------------------------------------------------------
truth_np, times = load_sequence(IC_KEY, START_INDEX, WINDOW_STEPS)
truth = torch.tensor(truth_np, dtype=torch.float32, device=device)

x_true0 = truth[0:1]

rng = np.random.default_rng(42)

bg_np = truth_np[0].copy()
bg_np[0] += rng.normal(0.0, PERT_ETA_STD, size=(NY, NX)).astype(np.float32)
bg_np[1] += rng.normal(0.0, PERT_UV_STD,  size=(NY, NX)).astype(np.float32)
bg_np[2] += rng.normal(0.0, PERT_UV_STD,  size=(NY, NX)).astype(np.float32)

x_bg = torch.tensor(bg_np[None, ...], dtype=torch.float32, device=device)

obs_mask_np = make_obs_mask(NY, NX, OBS_FRAC, seed=7)
obs_mask = torch.tensor(obs_mask_np[None, None, :, :], dtype=torch.float32, device=device)

obs = truth.clone()

if OBS_NOISE_ETA > 0.0:
    obs[:, 0] += OBS_NOISE_ETA * torch.randn_like(obs[:, 0])
if OBS_NOISE_UV > 0.0:
    obs[:, 1] += OBS_NOISE_UV * torch.randn_like(obs[:, 1])
    obs[:, 2] += OBS_NOISE_UV * torch.randn_like(obs[:, 2])

print("[Data] IC:", IC_KEY)
print("[Data] truth shape:", truth.shape)
print("[Data] obs fraction:", obs_mask_np.mean())

# ------------------------------------------------------------
# Optimize initial increment
# ------------------------------------------------------------
dx0 = torch.zeros_like(x_bg, requires_grad=True)

optimizer = torch.optim.Adam([dx0], lr=LR)

loss_history = []

for it in range(N_OPT):
    optimizer.zero_grad(set_to_none=True)

    x0 = x_bg + dx0
    pred_seq = rollout_model(model, x0, WINDOW_STEPS)

    loss_obs = torch.tensor(0.0, device=device)

    for k in range(1, WINDOW_STEPS + 1):
        pred_k = pred_seq[k]
        obs_k = obs[k:k+1]

        # sparse masked obs over all 3 variables
        diff = (pred_k - obs_k) * obs_mask
        loss_obs = loss_obs + (diff ** 2).mean()

    loss_bg = ((dx0) ** 2).mean()
    loss_sm = smoothness_loss(dx0)

    loss_uv_inc = (dx0[:, 1:] ** 2).mean()

    # loss = loss_obs + LAMBDA_BG * loss_bg + LAMBDA_SMOOTH * loss_sm
    loss = (
    loss_obs
    + LAMBDA_BG * loss_bg
    + LAMBDA_SMOOTH * loss_sm
    + LAMBDA_UV_INC * loss_uv_inc
)

    loss.backward()
    optimizer.step()

    loss_history.append([
        it,
        float(loss.detach().cpu()),
        float(loss_obs.detach().cpu()),
        float(loss_bg.detach().cpu()),
        float(loss_sm.detach().cpu()),
    ])

    if it % PRINT_EVERY == 0 or it == N_OPT - 1:
      print(
        f"[iter {it:04d}] "
        f"total={loss.item():.6e} "
        f"obs={loss_obs.item():.6e} "
        f"bg={loss_bg.item():.6e} "
        f"smooth={loss_sm.item():.6e} "
        f"uv_inc={loss_uv_inc.item():.6e}"
        f"geo_da={loss_geo_da.item():.6e}"
    )

# ------------------------------------------------------------
# Final rollout diagnostics
# ------------------------------------------------------------
with torch.no_grad():
    xb_roll = rollout_model(model, x_bg, WINDOW_STEPS)
    xa_roll = rollout_model(model, x_bg + dx0, WINDOW_STEPS)

xb_np = torch.cat(xb_roll, dim=0).detach().cpu().numpy()
xa_np = torch.cat(xa_roll, dim=0).detach().cpu().numpy()

rmse_rows = []

for k in range(WINDOW_STEPS + 1):
    row = {
        "step": k,
        "time_hr": k * DT_MACRO / 3600.0,
        "bg_eta_rmse": rmse_np(xb_np[k, 0], truth_np[k, 0]),
        "bg_u_rmse":   rmse_np(xb_np[k, 1], truth_np[k, 1]),
        "bg_v_rmse":   rmse_np(xb_np[k, 2], truth_np[k, 2]),
        "ana_eta_rmse": rmse_np(xa_np[k, 0], truth_np[k, 0]),
        "ana_u_rmse":   rmse_np(xa_np[k, 1], truth_np[k, 1]),
        "ana_v_rmse":   rmse_np(xa_np[k, 2], truth_np[k, 2]),
    }
    rmse_rows.append(row)

    print(
        f"step {k:02d} | "
        f"BG eta={row['bg_eta_rmse']:.4f}, u={row['bg_u_rmse']:.4f}, v={row['bg_v_rmse']:.4f} | "
        f"ANA eta={row['ana_eta_rmse']:.4f}, u={row['ana_u_rmse']:.4f}, v={row['ana_v_rmse']:.4f}"
    )

# ------------------------------------------------------------
# Save CSVs
# ------------------------------------------------------------
loss_csv = os.path.join(OUT_DIR, f"{IC_KEY}_loss_history.csv")
with open(loss_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["iter", "total", "obs", "bg", "smooth"])
    w.writerows(loss_history)

rmse_csv = os.path.join(OUT_DIR, f"{IC_KEY}_rmse_summary.csv")
with open(rmse_csv, "w", newline="") as f:
    keys = list(rmse_rows[0].keys())
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    for row in rmse_rows:
        w.writerow(row)

print("[Save] loss:", loss_csv)
print("[Save] rmse:", rmse_csv)

# ------------------------------------------------------------
# Plots
# ------------------------------------------------------------
loss_arr = np.array(loss_history)

plt.figure(figsize=(8, 5))
plt.plot(loss_arr[:, 0], loss_arr[:, 1], label="total")
plt.plot(loss_arr[:, 0], loss_arr[:, 2], label="obs")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title(f"ML-4DVAR loss | {IC_KEY}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{IC_KEY}_loss_curve.png"), dpi=160)
plt.close()

steps = np.array([r["step"] for r in rmse_rows])
hrs = np.array([r["time_hr"] for r in rmse_rows])

plt.figure(figsize=(8, 5))
plt.plot(hrs, [r["bg_eta_rmse"] for r in rmse_rows], marker="o", label="BG eta")
plt.plot(hrs, [r["ana_eta_rmse"] for r in rmse_rows], marker="o", label="ANA eta")
plt.xlabel("Time (hours)")
plt.ylabel("eta RMSE")
plt.title(f"eta RMSE before/after ML-4DVAR | {IC_KEY}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{IC_KEY}_eta_rmse_before_after.png"), dpi=160)
plt.close()

# field comparison at final time
k = WINDOW_STEPS

fig, axes = plt.subplots(3, 3, figsize=(14, 11))

titles = [
    "Truth eta", "Background eta", "Analysis eta",
    "Truth u",   "Background u",   "Analysis u",
    "Truth v",   "Background v",   "Analysis v",
]

fields = [
    truth_np[k, 0], xb_np[k, 0], xa_np[k, 0],
    truth_np[k, 1], xb_np[k, 1], xa_np[k, 1],
    truth_np[k, 2], xb_np[k, 2], xa_np[k, 2],
]

for ax, title, fld in zip(axes.flat, titles, fields):
    im = ax.imshow(fld, origin="lower")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{IC_KEY}_final_fields_truth_bg_analysis.png"), dpi=150)
plt.close()

# increment plot
dx0_np = dx0.detach().cpu().numpy()[0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for j, name in enumerate(["eta increment", "u increment", "v increment"]):
    im = axes[j].imshow(dx0_np[j], origin="lower")
    axes[j].set_title(name)
    plt.colorbar(im, ax=axes[j], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{IC_KEY}_analysis_increment.png"), dpi=150)
plt.close()

print("[Done] outputs saved to:", OUT_DIR)

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[Load] emulator: /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt
[Data] IC: wave_00
[Data] truth shape: torch.Size([5, 3, 128, 256])
[Data] obs fraction: 0.029571533
[iter 0000] total=1.096425e+02 obs=1.096425e+02 bg=0.000000e+00 smooth=0.000000e+00 uv_inc=0.000000e+00
[iter 0025] total=9.222035e+01 obs=9.211086e+01 bg=1.863579e-01 smooth=6.109188e-01 uv_inc=1.804983e-01
[iter 0050] total=7.908350e+01 obs=7.875780e+01 bg=5.719477e-01 smooth=1.865043e+00 uv_inc=5.332723e-01
[iter 0075] total=6.869653e+01 obs=6.814533e+01 bg=9.994766e-01 smooth=3.275577e+00 uv_inc=8.959577e-01
[iter 0100] total=6.088869e+01 obs=6.012956e+01 bg=1.418565e+00 smooth=4.673203e+00 uv_inc=1.225198e+00
[iter 0125] total=5.449859e+01 obs=5.353277e+01 bg=1.850283e+00 smooth=6.118584e+00 uv_inc=1.549347e+00
[iter 0150] total=4.947298e+01 obs=4.831146e+01 bg=2.277435e+00 smooth=7.551332e+00 uv_inc=1.

# ML DA for all cases

In [5]:
# ============================================================
# ml4dvar_lite_multicase_rescnn_1layer.py
# ------------------------------------------------------------
# Multi-case ML-4DVAR-lite validation for final 1-layer SWE
# emulator:
#
#   rescnn_b12_c96_lr1e4_t6_p4_keweak
#
# Runs several IC cases, saves:
#   - per-case loss history CSV
#   - per-case RMSE time-series CSV
#   - per-case loss curve
#   - per-case eta RMSE before/after plot
#   - per-case final field plots
#   - per-case analysis increment plots
#   - combined summary CSV
#
# Output:
#   /content/drive/MyDrive/AI_4DVAR2/ml4dvar_lite_multicase_outputs
# ============================================================

import os
import re
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
ROOT_IN  = "/content/drive/MyDrive/AI_4DVAR"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR2"

DATA_DIR = f"{ROOT_IN}/klein_ckpt_1L"

CKPT_PATH = (
    f"{ROOT_OUT}/training_runs/"
    f"rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt"
)

OUT_DIR = f"{ROOT_OUT}/ml4dvar_lite_multicase_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
H = 1000.0
DT_MACRO = 200.0 * 30.0

NX = 256
NY = 128

#-------------------------------------------------------------
# For geostrophic balance
#-------------------------------------------------------------
G = 9.81
Lx = 2.0e7
Ly = 8.0e6
DX = Lx / NX
DY = Ly / NY
FMIN = 2.0e-5

# LAMBDA_GEO_DA = 1e-4   # start weak
LAMBDA_GEO_DA = 1e-2   # start weak
# ------------------------------------------------------------
# USER CONTROLS
# ------------------------------------------------------------
CASE_LIST = [
    "gauss_00",
    "test_modon_00",
    "test_rh_00",
    "wave_00",
]

START_INDEX = 0
WINDOW_STEPS = 4

OBS_FRAC = 0.03
OBS_NOISE_ETA = 0.0
OBS_NOISE_UV  = 0.0

N_OPT = 250
LR = 2e-2

LAMBDA_BG = 0.10
LAMBDA_SMOOTH = 1e-3
LAMBDA_UV_INC = 0.50

PERT_ETA_STD = 2.0
PERT_UV_STD  = 0.2

PRINT_EVERY = 25

# ------------------------------------------------------------
# Model definition
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=12):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]

        blocks = []
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        # Present in trained checkpoint. Not used by rollout,
        # but needed for clean state_dict loading.
        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def forward_step(self, x):
        feat = self.encode(x)
        xdot = self.grid_tendency(feat)
        return x + DT_MACRO * xdot

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_sequence(ic_key, start_index=0, window_steps=4):
    ic_dir = os.path.join(DATA_DIR, ic_key)
    files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))

    if len(files) < start_index + window_steps + 1:
        raise RuntimeError(f"Not enough snapshots for {ic_key}")

    seq = []
    times = []

    for f in files[start_index:start_index + window_steps + 1]:
        z = np.load(f)
        state = np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32)
        seq.append(state)
        times.append(float(z["t"]))

    seq = np.stack(seq, axis=0)
    times = np.array(times)

    return seq, times

def make_obs_mask(ny, nx, obs_frac, seed=123):
    rng = np.random.default_rng(seed)
    mask = rng.random((ny, nx)) < obs_frac
    return mask.astype(np.float32)

def ddx_torch(a, dx):
    # periodic in x
    return (torch.roll(a, shifts=-1, dims=-1) - torch.roll(a, shifts=1, dims=-1)) / (2.0 * dx)

def ddy_torch(a, dy):
    out = torch.zeros_like(a)
    out[..., 1:-1, :] = (a[..., 2:, :] - a[..., :-2, :]) / (2.0 * dy)
    out[..., 0, :] = (a[..., 1, :] - a[..., 0, :]) / dy
    out[..., -1, :] = (a[..., -1, :] - a[..., -2, :]) / dy
    return out

def make_f_field(ny, nx, device):
    # y in [-pi/2, pi/2], simple beta-plane/Klein latitude proxy
    y = torch.linspace(-0.5 * np.pi, 0.5 * np.pi, ny, device=device)
    f = torch.sin(y).view(1, 1, ny, 1).repeat(1, 1, 1, nx)

    # scale to avoid unrealistically large geostrophic velocity near equator
    # If your FD f field is saved and you prefer exact values, we can replace this.
    f0 = 1.0e-4
    return f0 * f

def geostrophic_balance_loss(state):
    """
    state: [1, 3, NY, NX], channels [eta, u, v]
    """
    eta = state[:, 0:1]
    u = state[:, 1:2]
    v = state[:, 2:3]

    eta_x = ddx_torch(eta, DX)
    eta_y = ddy_torch(eta, DY)

    f = make_f_field(NY, NX, state.device)
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    f_safe = sign * torch.clamp(torch.abs(f), min=FMIN)

    u_geo = -(G / f_safe) * eta_y
    v_geo =  (G / f_safe) * eta_x

    return ((u - u_geo) ** 2).mean() + ((v - v_geo) ** 2).mean()

def smoothness_loss(dx):
    ddx = dx[:, :, :, 1:] - dx[:, :, :, :-1]
    ddy = dx[:, :, 1:, :] - dx[:, :, :-1, :]
    return (ddx ** 2).mean() + (ddy ** 2).mean()

def rmse_np(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def rollout_model(model, x0, steps):
    xs = [x0]
    x = x0
    for _ in range(steps):
        x = model.forward_step(x)
        xs.append(x)
    return xs

def pct_improve(bg, ana):
    if abs(bg) < 1e-12:
        return 0.0
    return 100.0 * (bg - ana) / bg

def combined_rmse(eta, u, v):
    return float(np.sqrt(eta ** 2 + u ** 2 + v ** 2))

# ------------------------------------------------------------
# Load emulator
# ------------------------------------------------------------
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(CKPT_PATH)

model = MultiStepContinuousResCNNModel(
    feat_ch=96,
    hidden=192,
    n_blocks=12,
).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

for p in model.parameters():
    p.requires_grad_(False)

print("[Load] emulator:", CKPT_PATH)

# ------------------------------------------------------------
# Run one case
# ------------------------------------------------------------
def run_one_case(ic_key):
    print("\n" + "=" * 80)
    print(f"[Case] {ic_key}")
    print("=" * 80)

    truth_np, times = load_sequence(ic_key, START_INDEX, WINDOW_STEPS)
    truth = torch.tensor(truth_np, dtype=torch.float32, device=device)

    rng = np.random.default_rng(42)

    bg_np = truth_np[0].copy()
    bg_np[0] += rng.normal(0.0, PERT_ETA_STD, size=(NY, NX)).astype(np.float32)
    bg_np[1] += rng.normal(0.0, PERT_UV_STD,  size=(NY, NX)).astype(np.float32)
    bg_np[2] += rng.normal(0.0, PERT_UV_STD,  size=(NY, NX)).astype(np.float32)

    x_bg = torch.tensor(bg_np[None, ...], dtype=torch.float32, device=device)

    obs_mask_np = make_obs_mask(NY, NX, OBS_FRAC, seed=7)
    obs_mask = torch.tensor(obs_mask_np[None, None, :, :], dtype=torch.float32, device=device)

    obs = truth.clone()

    if OBS_NOISE_ETA > 0.0:
        obs[:, 0] += OBS_NOISE_ETA * torch.randn_like(obs[:, 0])
    if OBS_NOISE_UV > 0.0:
        obs[:, 1] += OBS_NOISE_UV * torch.randn_like(obs[:, 1])
        obs[:, 2] += OBS_NOISE_UV * torch.randn_like(obs[:, 2])

    print("[Data] truth shape:", truth.shape)
    print("[Data] obs fraction:", obs_mask_np.mean())

    dx0 = torch.zeros_like(x_bg, requires_grad=True)
    optimizer = torch.optim.Adam([dx0], lr=LR)

    loss_history = []

    for it in range(N_OPT):
        optimizer.zero_grad(set_to_none=True)

        x0 = x_bg + dx0
        pred_seq = rollout_model(model, x0, WINDOW_STEPS)

        loss_obs = torch.tensor(0.0, device=device)

        for k in range(1, WINDOW_STEPS + 1):
            pred_k = pred_seq[k]
            obs_k = obs[k:k+1]

            diff = (pred_k - obs_k) * obs_mask
            loss_obs = loss_obs + (diff ** 2).mean()

        loss_bg = (dx0 ** 2).mean()
        loss_sm = smoothness_loss(dx0)
        loss_uv_inc = (dx0[:, 1:] ** 2).mean()

        # x0_analysis = x_bg + dx0
        # loss_geo_da = geostrophic_balance_loss(x0_analysis)
        loss_geo_da = geostrophic_balance_loss(dx0)

        loss = (
          loss_obs
          + LAMBDA_BG * loss_bg
          + LAMBDA_SMOOTH * loss_sm
          + LAMBDA_UV_INC * loss_uv_inc
          + LAMBDA_GEO_DA * loss_geo_da
        )



        loss.backward()
        optimizer.step()

        loss_history.append([
            it,
            float(loss.detach().cpu()),
            float(loss_obs.detach().cpu()),
            float(loss_bg.detach().cpu()),
            float(loss_sm.detach().cpu()),
            float(loss_uv_inc.detach().cpu()),
        ])

        if it % PRINT_EVERY == 0 or it == N_OPT - 1:
            print(
                f"[iter {it:04d}] "
                f"total={loss.item():.6e} "
                f"obs={loss_obs.item():.6e} "
                f"bg={loss_bg.item():.6e} "
                f"smooth={loss_sm.item():.6e} "
                f"uv_inc={loss_uv_inc.item():.6e}"
            )

    with torch.no_grad():
        xb_roll = rollout_model(model, x_bg, WINDOW_STEPS)
        xa_roll = rollout_model(model, x_bg + dx0, WINDOW_STEPS)

    xb_np = torch.cat(xb_roll, dim=0).detach().cpu().numpy()
    xa_np = torch.cat(xa_roll, dim=0).detach().cpu().numpy()
    dx0_np = dx0.detach().cpu().numpy()[0]

    rmse_rows = []

    for k in range(WINDOW_STEPS + 1):
        bg_eta = rmse_np(xb_np[k, 0], truth_np[k, 0])
        bg_u   = rmse_np(xb_np[k, 1], truth_np[k, 1])
        bg_v   = rmse_np(xb_np[k, 2], truth_np[k, 2])

        ana_eta = rmse_np(xa_np[k, 0], truth_np[k, 0])
        ana_u   = rmse_np(xa_np[k, 1], truth_np[k, 1])
        ana_v   = rmse_np(xa_np[k, 2], truth_np[k, 2])

        row = {
            "case": ic_key,
            "step": k,
            "time_hr": k * DT_MACRO / 3600.0,
            "bg_eta_rmse": bg_eta,
            "bg_u_rmse": bg_u,
            "bg_v_rmse": bg_v,
            "bg_total_rmse": combined_rmse(bg_eta, bg_u, bg_v),
            "ana_eta_rmse": ana_eta,
            "ana_u_rmse": ana_u,
            "ana_v_rmse": ana_v,
            "ana_total_rmse": combined_rmse(ana_eta, ana_u, ana_v),
        }

        rmse_rows.append(row)

        print(
            f"step {k:02d} | "
            f"BG eta={bg_eta:.4f}, u={bg_u:.4f}, v={bg_v:.4f}, total={row['bg_total_rmse']:.4f} | "
            f"ANA eta={ana_eta:.4f}, u={ana_u:.4f}, v={ana_v:.4f}, total={row['ana_total_rmse']:.4f}"
        )

    # --------------------------------------------------------
    # Save per-case CSVs
    # --------------------------------------------------------
    loss_csv = os.path.join(OUT_DIR, f"{ic_key}_loss_history.csv")
    with open(loss_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["iter", "total", "obs", "bg", "smooth", "uv_inc"])
        w.writerows(loss_history)

    rmse_csv = os.path.join(OUT_DIR, f"{ic_key}_rmse_summary.csv")
    with open(rmse_csv, "w", newline="") as f:
        keys = list(rmse_rows[0].keys())
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in rmse_rows:
            w.writerow(row)

    # --------------------------------------------------------
    # Plots
    # --------------------------------------------------------
    loss_arr = np.array(loss_history)

    plt.figure(figsize=(8, 5))
    plt.plot(loss_arr[:, 0], loss_arr[:, 1], label="total")
    plt.plot(loss_arr[:, 0], loss_arr[:, 2], label="obs")
    plt.yscale("log")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.title(f"ML-4DVAR loss | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{ic_key}_loss_curve.png"), dpi=160)
    plt.close()

    hrs = np.array([r["time_hr"] for r in rmse_rows])

    plt.figure(figsize=(8, 5))
    plt.plot(hrs, [r["bg_eta_rmse"] for r in rmse_rows], marker="o", label="BG eta")
    plt.plot(hrs, [r["ana_eta_rmse"] for r in rmse_rows], marker="o", label="ANA eta")
    plt.xlabel("Time (hours)")
    plt.ylabel("eta RMSE")
    plt.title(f"eta RMSE before/after ML-4DVAR | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{ic_key}_eta_rmse_before_after.png"), dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(hrs, [r["bg_total_rmse"] for r in rmse_rows], marker="o", label="BG total")
    plt.plot(hrs, [r["ana_total_rmse"] for r in rmse_rows], marker="o", label="ANA total")
    plt.xlabel("Time (hours)")
    plt.ylabel("Combined RMSE")
    plt.title(f"Combined RMSE before/after ML-4DVAR | {ic_key}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{ic_key}_total_rmse_before_after.png"), dpi=160)
    plt.close()

    k = WINDOW_STEPS

    fig, axes = plt.subplots(3, 3, figsize=(14, 11))

    titles = [
        "Truth eta", "Background eta", "Analysis eta",
        "Truth u",   "Background u",   "Analysis u",
        "Truth v",   "Background v",   "Analysis v",
    ]

    fields = [
        truth_np[k, 0], xb_np[k, 0], xa_np[k, 0],
        truth_np[k, 1], xb_np[k, 1], xa_np[k, 1],
        truth_np[k, 2], xb_np[k, 2], xa_np[k, 2],
    ]

    for ax, title, fld in zip(axes.flat, titles, fields):
        im = ax.imshow(fld, origin="lower")
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{ic_key}_final_fields_truth_bg_analysis.png"), dpi=150)
    plt.close()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for j, name in enumerate(["eta increment", "u increment", "v increment"]):
        im = axes[j].imshow(dx0_np[j], origin="lower")
        axes[j].set_title(name)
        plt.colorbar(im, ax=axes[j], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{ic_key}_analysis_increment.png"), dpi=150)
    plt.close()

    final = rmse_rows[-1]

    summary_row = {
        "case": ic_key,
        "window_steps": WINDOW_STEPS,
        "obs_frac": OBS_FRAC,
        "n_opt": N_OPT,
        "lambda_bg": LAMBDA_BG,
        "lambda_smooth": LAMBDA_SMOOTH,
        "lambda_uv_inc": LAMBDA_UV_INC,
        "lr": LR,

        "final_loss_total": loss_history[-1][1],
        "final_loss_obs": loss_history[-1][2],
        "final_loss_bg": loss_history[-1][3],
        "final_loss_smooth": loss_history[-1][4],
        "final_loss_uv_inc": loss_history[-1][5],

        "bg_eta_final": final["bg_eta_rmse"],
        "bg_u_final": final["bg_u_rmse"],
        "bg_v_final": final["bg_v_rmse"],
        "bg_total_final": final["bg_total_rmse"],

        "ana_eta_final": final["ana_eta_rmse"],
        "ana_u_final": final["ana_u_rmse"],
        "ana_v_final": final["ana_v_rmse"],
        "ana_total_final": final["ana_total_rmse"],

        "eta_improve_pct": pct_improve(final["bg_eta_rmse"], final["ana_eta_rmse"]),
        "u_improve_pct": pct_improve(final["bg_u_rmse"], final["ana_u_rmse"]),
        "v_improve_pct": pct_improve(final["bg_v_rmse"], final["ana_v_rmse"]),
        "total_improve_pct": pct_improve(final["bg_total_rmse"], final["ana_total_rmse"]),

        "dx0_eta_rms": float(np.sqrt(np.mean(dx0_np[0] ** 2))),
        "dx0_u_rms": float(np.sqrt(np.mean(dx0_np[1] ** 2))),
        "dx0_v_rms": float(np.sqrt(np.mean(dx0_np[2] ** 2))),
        "dx0_total_rms": float(np.sqrt(np.mean(dx0_np ** 2))),
    }

    print("[Save] loss:", loss_csv)
    print("[Save] rmse:", rmse_csv)

    return summary_row

# ------------------------------------------------------------
# Main multicase loop
# ------------------------------------------------------------
summary_rows = []

for ic_key in CASE_LIST:
    try:
        row = run_one_case(ic_key)
        summary_rows.append(row)
    except Exception as e:
        print(f"[ERROR] case {ic_key} failed: {repr(e)}")

# ------------------------------------------------------------
# Save combined summary
# ------------------------------------------------------------
summary_csv = os.path.join(OUT_DIR, "ml4dvar_multicase_summary.csv")

if len(summary_rows) > 0:
    with open(summary_csv, "w", newline="") as f:
        keys = list(summary_rows[0].keys())
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for row in summary_rows:
            w.writerow(row)

    print("\n[Done] combined summary:", summary_csv)
else:
    print("\n[Done] no successful cases.")

print("[Done] outputs saved to:", OUT_DIR)

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
[Load] emulator: /content/drive/MyDrive/AI_4DVAR2/training_runs/rescnn_b12_c96_lr1e4_t6_p4_keweak/final_model.pt

[Case] gauss_00
[Data] truth shape: torch.Size([5, 3, 128, 256])
[Data] obs fraction: 0.029571533
[iter 0000] total=5.537133e-01 obs=5.537133e-01 bg=0.000000e+00 smooth=0.000000e+00 uv_inc=0.000000e+00
[iter 0025] total=3.414993e-01 obs=3.113484e-01 bg=6.995489e-02 smooth=1.485917e-01 uv_inc=3.723003e-02
[iter 0050] total=2.381517e-01 obs=1.917841e-01 bg=1.240135e-01 smooth=2.862880e-01 uv_inc=5.265426e-02
[iter 0075] total=1.902585e-01 obs=1.386762e-01 bg=1.385390e-01 smooth=3.606091e-01 uv_inc=5.716908e-02
[iter 0100] total=1.661101e-01 obs=1.122987e-01 bg=1.429791e-01 smooth=4.109497e-01 uv_inc=5.864265e-02
[iter 0125] total=1.520423e-01 obs=9.727344e-02 bg=1.444577e-01 smooth=4.454644e-01 uv_inc=5.867380e-02
[iter 0150] total=1.427256e-01 obs=8.787581e-02 bg=1.445245e-01 smooth=4.689619e-01 uv_inc=5.76